# ARISE — 04 · Full Pipeline, Evaluation & Comparison

Putting it together (paper **Algorithm 1**). The final score fuses both modules (Eq. 12):

$$\text{score}(v_i) = (1-\alpha)\cdot \text{score\_t}(v_i) + \alpha\cdot \text{score\_a}(v_i),\quad \alpha=0.8$$

We evaluate with **AUC**, **AUPRC**, **Precision@K** (Sec. V-A4) and compare against baselines.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # make the `arise` package importable
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
from arise.data import load_dataset, inject_anomalies, PAPER_INJECTION
from arise.pipeline import run_arise

graph = load_dataset("cora")
graph = inject_anomalies(graph, seed=42, **PAPER_INJECTION["Cora"])
result, model = run_arise(graph, alpha=0.8, epochs=60, lr=0.003,
                          weight_decay=1e-5, attr_rounds=4, seed=42,
                          return_model=True, verbose=True)

## 1. Headline metrics

In [ ]:
import json
print(json.dumps(result["metrics"]["overall"], indent=2))
print("topology-only AUC :", result["metrics"]["topology_only"]["auc"])
print("attribute-only AUC:", result["metrics"]["attribute_only"]["auc"])

## 2. Compare against baselines (LOF, attribute residual, random)

In [ ]:
from arise.baselines import run_baselines
import numpy as np

baselines = run_baselines(graph)
print(f"{'method':<22}{'AUC':>8}{'AUPRC':>8}")
print(f"{'ARISE (full)':<22}{result['metrics']['overall']['auc']:>8}{result['metrics']['overall']['auprc']:>8}")
for name, b in baselines.items():
    print(f"{name:<22}{b['metrics']['auc']:>8}{b['metrics']['auprc']:>8}")

In [ ]:
methods = ["ARISE (full)", "ARISE (attr-only)", "ARISE (topo-only)"] + list(baselines.keys())
aucs = [result['metrics']['overall']['auc'], result['metrics']['attribute_only']['auc'],
        result['metrics']['topology_only']['auc']] + [b['metrics']['auc'] for b in baselines.values()]
colors = ["#ff5c5c"] + ["#6ea8ff"]*(len(methods)-1)
plt.figure(figsize=(9,4))
plt.bar(methods, aucs, color=colors)
plt.ylabel("AUC"); plt.xticks(rotation=30, ha="right"); plt.title("AUC comparison (Cora)")
plt.ylim(0.4,1.0); plt.tight_layout(); plt.show()

## 3. ROC & PR curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
final = np.asarray(result["scores"]["final"]); labels = np.asarray(result["labels"])
fig, ax = plt.subplots(1,2, figsize=(12,4))
fpr,tpr,_ = roc_curve(labels, final); ax[0].plot(fpr,tpr,label="ARISE",color="#ff5c5c")
ax[0].plot([0,1],[0,1],"--",color="#888"); ax[0].set_title("ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
prec,rec,_ = precision_recall_curve(labels, final); ax[1].plot(rec,prec,color="#ff5c5c")
ax[1].set_title("Precision-Recall"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
plt.show()

## 4. Trade-off parameter α (reproduces paper Fig. 6)
Using topology-only at α=0 and attribute-only at α=1; the best is in between.

In [ ]:
norm_t = np.asarray(result["scores"]["topology_norm"]); norm_a = np.asarray(result["scores"]["attribute_norm"])
from arise.metrics import auc as auc_fn
alphas = np.linspace(0,1,11)
sweep = [auc_fn(labels, (1-a)*norm_t + a*norm_a) for a in alphas]
plt.figure(figsize=(7,4))
plt.plot(alphas, sweep, "-o", color="#6ea8ff")
plt.axvline(0.8, color="#ff5c5c", ls="--", label="α = 0.8")
plt.xlabel("α"); plt.ylabel("AUC"); plt.title("Effect of fusion weight α"); plt.legend(); plt.show()

## Summary
- Full ARISE beats the attribute-only and topology-only ablations and the classic baselines.
- The α sweep confirms that **combining** both modules (α≈0.8) beats using either alone — matching the paper's finding.

➡️ Next: **05** produces the rich visualisations and exports the JSON consumed by the React dashboard.